In [ ]:
# import libraries
try:
  # %tensorflow_version only exists in Colab.
  !pip install tf-nightly
except Exception:
  pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
# Read in training and testing data
col_names = ['class', 'message']
train_df = pd.read_csv(train_file_path, sep='\t', names=col_names)
test_df = pd.read_csv(test_file_path, sep='\t', names=col_names)

# Encode ham as 0 and spam as 1
train_df['class'] = pd.factorize(train_df['class'])[0]
test_df['class'] = pd.factorize(test_df['class'])[0]

In [ ]:
# Extract the label columns and create Tensorflow dataset objects to be used as
# an input for the model
train_labels = train_df["class"].values
train_dataset = tf.data.Dataset.from_tensor_slices(
    (train_df['message'].values, train_labels)
)

test_labels = test_df['class'].values
test_dataset = tf.data.Dataset.from_tensor_slices(
    (test_df['message'].values, test_labels)
)

In [ ]:
# Shuffle the data for training and create batches of (text, label) pairs
BUFFER_SIZE = 100
BATCH_SIZE = 32
train_dataset = train_dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
# TextVectorization layer to transform strings into token indices
encoder = tf.keras.layers.TextVectorization(
    max_tokens=1000,
    output_sequence_length=1000,
)

# adapt method sets the model's vocabulary
encoder.adapt(train_dataset.map(lambda text, label: text))

In [ ]:
vocab = np.array(encoder.get_vocabulary())
vocab[:20]

In [ ]:
# Create the Recurrent Neural Network
model = tf.keras.Sequential([
    encoder,
    tf.keras.layers.Embedding(len(encoder.get_vocabulary()), 64, mask_zero=True),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64,  return_sequences=True)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1)
])

model.compile(loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              optimizer=tf.keras.optimizers.Adam(1e-4),
              metrics=['accuracy'])

In [ ]:
# Train the model over 10 epochs
history = model.fit(train_dataset, epochs=10,
                    validation_data=test_dataset,
                    validation_steps=30)

In [ ]:
def predict_message(pred_text):
    """
    Predicts messages as normal or spam based on the model.
    """
    # Convert pred_text to tf.constant to be allowed as input in the model
    pred_text_tensor = tf.constant([pred_text]) 
    predictions = model.predict(pred_text_tensor)
    prediction = predictions[0][0]
    return [prediction, "ham" if prediction < 0.5 else "spam"]

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Testing function for the model
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False
test_predictions()
